# Train Bilingual Conformer ASR in Google Colab

Run the cells from top to bottom. Before training, choose **Runtime → Change runtime type → GPU**. Dataset and results use Colab's temporary `/content` disk, so Google Drive storage is not required. Files in `/content` are deleted when the runtime resets; download checkpoints before disconnecting if you need to resume later.

**Dataset required:** the course audio and manifests are not stored in this GitHub repository. Delta's `/work/nvme/bjdh/shared/coding4` directory is not available from Colab. After transferring the course-provided dataset to this runtime, place its `dump` directory at `/content/dump` (it should include `raw/train_bilingual/data.json`, `raw/dev_bilingual/data.json`, and referenced audio). Follow the course's data access and sharing rules.

The Delta `sbatch` script is not used here. Training runs directly through `train.py`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_DIR = Path('/content/Bilingual-Conformer-ASR')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Yen-hub/Bilingual-Conformer-ASR.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', Path.cwd())

In [ ]:
# Install Python packages without replacing Colab's CUDA-enabled PyTorch.
%pip install -r requirements.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab choose Runtime → Change runtime type → GPU, reconnect, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
import shutil

disk = shutil.disk_usage('/content')
print(f'Colab disk free: {disk.free / (1024**3):.1f} GiB')
DATA_DUMP = Path('/content/dump')
required_manifests = [
    DATA_DUMP / 'raw/train_bilingual/data.json',
    DATA_DUMP / 'raw/dev_bilingual/data.json',
]
missing = [str(p) for p in required_manifests if not p.is_file()]
if missing:
    raise FileNotFoundError('Course dataset not found. Transfer it into this Colab runtime at ' + str(DATA_DUMP) + '. Missing: ' + ', '.join(missing))

repo_dump = REPO_DIR / 'dump'
if repo_dump.is_symlink():
    repo_dump.unlink()
elif repo_dump.exists():
    raise FileExistsError(f'{repo_dump} exists; move it before linking the runtime dataset.')
repo_dump.symlink_to(DATA_DUMP, target_is_directory=True)
print('Dataset linked:', repo_dump, '→', DATA_DUMP)

## Train

The recipe configuration sets the model and epoch count. Outputs are written to the temporary Colab disk. If the runtime resets, files there are lost; download a checkpoint before disconnecting, then upload it in a later session and set `RESUME_FROM` to its path. The first run can take a while; its epoch timing will help estimate runtime and compute-unit use.

In [ ]:
OUTPUT_ROOT = Path('/content/coding4_colab_results')
RESUME_FROM = ''  # Optional: full path to a saved .pth checkpoint
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

train_command = [
    sys.executable, 'train.py',
    '--tag', 'multilingual',
    '--mode', 'multilingual',
    '--train_json', 'dump/raw/train_bilingual/data.json',
    '--valid_json', 'dump/raw/dev_bilingual/data.json',
    '--out_dir', str(OUTPUT_ROOT),
    '--nworkers', '2',
]
if RESUME_FROM.strip():
    train_command += ['--resume', RESUME_FROM.strip()]
print('Running:', ' '.join(train_command))
subprocess.run(train_command, cwd=REPO_DIR, check=True)

## Find the saved checkpoint

The best checkpoint is saved under `/content/coding4_colab_results/exp/train_multilingual/ckpts`. Download it before the runtime resets if you want to resume training in a later session. Keep the Delta job queued separately if you still want it to run there.

In [ ]:
CHECKPOINT_DIR = OUTPUT_ROOT / 'exp/train_multilingual/ckpts'
checkpoints = sorted(CHECKPOINT_DIR.glob('*.pth')) if CHECKPOINT_DIR.exists() else []
for path in checkpoints:
    print(path)
if not checkpoints:
    print('No checkpoint saved yet.')

# In Colab, uncomment the next two lines to download the best checkpoint to your computer.
# from google.colab import files
# files.download(str(checkpoints[-1])) if checkpoints else None